# Setting up the word aligner

In [ ]:
# Set this to the directory to where you have installed glaux-nlp
import sys
sys.path.append('glaux-nlp')

In [ ]:
import ast
import pandas as pd
from alignment.WordAligner import WordAligner
from alignment.Scorer import Scorer
from vectors.VectorExtractor import VectorExtractor

In [ ]:
training_data={'input_file':'Data/NT_sentences_reduced.txt','docbin':'Data/NT_analysis.docbin','grc_lemmas':'Data/NT_lemmas.txt','grc_tags':'Data/NT_tags.txt'}
gold_data={'input_file':'Data/ugarit_gold_sentences_reduced.txt','docbin':'Data/ugarit_gold_analysis.docbin','grc_lemmas':'Data/ugarit_gold_lemmas.txt','grc_tags':'Data/ugarit_gold_tags.txt'}

In [ ]:
# Decrease batch size if this is too much for your memory
wa = WordAligner(lexicon_file='Data/grc_english_megalexicon.txt',vectors_name='conceptnet-numberbatch-17-06-300',training_data=training_data,gold_data=gold_data,freq_matrix='Data/freq_matrix.npz',freq_matrix_data='Data/freq_matrix_data.pkl',phrase_model='Data/phrase_model.pkl',language_model='UGARIT/grc-alignment',batch_size=100)

# Results: general overview (5.1)

Training the LTR model and printing metrics

In [ ]:
wa.train()

Prediction accuracy by word class

In [ ]:
grc_pos = {}
for sent_no, sent in enumerate(wa.sentences_gold):
    for word_no, pos in enumerate(sent['grc_pos']):
        grc_pos[f'{sent_no}_{word_no}'] = pos

In [ ]:
pos_metrics = {}
for pos in set(grc_pos.values()):
    scorer = Scorer(wa.results,wa.sentences_gold)
    accuracy, correct, total = scorer.get_accuracy(wa.threshold,pos,grc_pos)
    pos_metrics[pos] = [correct,total]
pos_metrics_red = {}
for k, v in pos_metrics.items():
    if k == 'noun' or k == 'adjective':
        pos_metrics_red[k] = v
    elif k == 'verb' or k == 'participle' or k == 'infinitive':
        metrics = pos_metrics_red.get('verb',[0,0])
        metrics[0] += v[0]
        metrics[1] += v[1]
        pos_metrics_red['verb'] = metrics
    else:
        metrics = pos_metrics_red.get('other',[0,0])
        metrics[0] += v[0]
        metrics[1] += v[1]
        pos_metrics_red['other'] = metrics
for k, v in pos_metrics_red.items():
    print(f'{k}\t{v[0]}\t{v[1]}\t{v[0]/v[1]}')
print('---')
for k, v in pos_metrics.items():
    print(f'{k}\t{v[0]}\t{v[1]}\t{v[0]/v[1]}')

Comparing part-of-speech results across models

In [ ]:
wa.results_df['INDEX'] = wa.results_df['SENT'].astype(str)+'_'+wa.results_df['GRC_INDEX'].astype(str)
wa.results_df['GRC_POS'] = wa.results_df['INDEX'].map(grc_pos)
other_method_results = {}
with open('Data/other_method_results.tsv',encoding='utf8') as infile:
    lines = infile.readlines()
    for line in lines[1:len(lines)]:
        sl = line.strip('\n').split('\t')
        argmax_results = other_method_results.get('argmax',{})
        entmax_results = other_method_results.get('entmax',{})
        argmax_results[f'{sl[0]}_{sl[1]}'] = ast.literal_eval(sl[4])
        entmax_results[f'{sl[0]}_{sl[1]}'] = ast.literal_eval(sl[7])
        other_method_results['argmax'] = argmax_results
        other_method_results['entmax'] = entmax_results
wa.results_df['ARGMAX'] = wa.results_df['INDEX'].map(other_method_results['argmax'])
wa.results_df['ENTMAX'] = wa.results_df['INDEX'].map(other_method_results['entmax'])
wa.results_df['ARGMAX_CORRECT'] = wa.results_df['GOLD'] == wa.results_df['ARGMAX']
wa.results_df['ENTMAX_CORRECT'] = wa.results_df['GOLD'] == wa.results_df['ENTMAX']

In [ ]:
print(len(wa.results_df[(wa.results_df['GRC_POS']=='article')&(wa.results_df['CORRECT']==0)&(wa.results_df['ARGMAX_CORRECT']==1)]))
print(len(wa.results_df[(wa.results_df['GRC_POS']=='article')&(wa.results_df['CORRECT']==0)&(wa.results_df['ARGMAX_CORRECT']==1)&(wa.results_df['UNALIGNED']==True)]))
print(len(wa.results_df[(wa.results_df['GRC_POS']=='article')&(wa.results_df['CORRECT']==0)&(wa.results_df['ARGMAX_CORRECT']==1)&(wa.results_df['UNALIGNED']==False)&(wa.results_df['SCORE']<wa.threshold)]))

In [ ]:
print(len(wa.results_df[(wa.results_df['GRC_POS']=='particle')&(wa.results_df['CORRECT']==0)&(wa.results_df['ARGMAX_CORRECT']==1)]))
print(len(wa.results_df[(wa.results_df['GRC_POS']=='particle')&(wa.results_df['CORRECT']==0)&(wa.results_df['ARGMAX_CORRECT']==1)&(wa.results_df['UNALIGNED']==True)]))
print(len(wa.results_df[(wa.results_df['GRC_POS']=='particle')&(wa.results_df['CORRECT']==0)&(wa.results_df['ARGMAX_CORRECT']==1)&(wa.results_df['UNALIGNED']==False)&(wa.results_df['SCORE']<wa.threshold)]))

In [ ]:
wa.results_df[(wa.results_df['GRC_POS']=='interjection')&(wa.results_df['CORRECT']==0)&(wa.results_df['ARGMAX_CORRECT']==1)]

# Results: transformer model (5.2)

In [ ]:
wa.extractor = VectorExtractor(transformer_path='glaux-nlp/PhilBerta-WordAlignment',tokenizer_add_prefix_space=True,layers=[8])
wa.vectors_en_td, wa.vectors_grc_td = wa.get_bilingual_embeddings(wa.sentences_train, batch_size=100)
wa.vectors_en_gold, wa.vectors_grc_gold = wa.get_bilingual_embeddings(wa.sentences_gold, batch_size=100)

In [ ]:
wa.train(build_datasets=True)

# Results: features (5.3)

In [ ]:
feats = ['COSINE','PMI','REL_FREQ','POS_DIFF','IN_LEXICON','COSINE_STATIC','COSINE_LEXICON','IS_PHRASE']
for feat in feats:
    print(feat)
    wa.train(ignore_features=[feat])
    print('---')

In [ ]:
wa.train(ignore_features=['PMI','REL_FREQ','POS_DIFF','IN_LEXICON','COSINE_STATIC','COSINE_LEXICON','IS_PHRASE'])

In [ ]:
import shap
wa.train()
gold = pd.DataFrame(wa.gold_data,columns=['GROUP','SENT','GRC','EN','GRC_INDEX','EN_INDICES','ALIGNED','COSINE','PMI','REL_FREQ','POS_DIFF','IN_LEXICON','COSINE_STATIC','COSINE_LEXICON','IS_PHRASE'])
explainer = shap.TreeExplainer(wa.model)
shap_values = explainer(gold.copy().drop(columns=['GROUP','SENT','GRC','EN','GRC_INDEX','EN_INDICES','ALIGNED']))
shap_values.feature_names = ['CtxEmb','PMI','RelFreq','Posit','InLex','StatEmb','LexSim','Phrase']
shap.plots.beeswarm(shap_values)